# Data governance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/data-governance.ipynb)

Governance is the set of ownership, discovery, protection, and lifecycle controls that keep data trusted, secure, and correctly used as a platform grows to many teams. This notebook covers the concepts interviewers probe, the platform syntax for Unity Catalog, Snowflake, BigQuery, and Lake Formation, and runnable DuckDB and standard-library demos of contracts, masking, row filters, tokenization, PII discovery, and policy tests.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is data governance and why does it matter?

Governance makes data discoverable, trusted, secure, and correctly used. Without it, definitions diverge, access becomes risky, and compliance gaps appear; with it, self-service analytics gets faster and safer. The four pillars interviewers expect:

```text
1. Ownership   - every dataset has an accountable owner and a steward.
2. Discovery   - catalog, glossary, lineage, clear access-request paths.
3. Protection  - classification, least privilege, masking, encryption, audit.
4. Lifecycle   - quality SLAs, retention, archival, safe deprecation.
```

Governance runs as a loop: **define** the rule and name an owner, **enforce** it in the platform (not a wiki), **monitor** whether it holds, **review** it on a schedule. Example: a new `customer_email` column is tagged `pii` with 30-day raw retention (define); the engine attaches a mask and a daily retention job runs (enforce); a weekly report lists unclassified columns and failed retention runs (monitor); the tag is re-confirmed quarterly (review).

Common mistake: treating governance as documentation. A catalog entry saying "PII, restricted" changes nothing if the engine still returns the raw email. The honest test: can an ordinary analyst read the protected value right now?

Rules are easiest to enforce when stored as data that CI and the catalog both read:

```json
{
  "policy_id": "retention.raw_events",
  "owner": "platform-data@example.com",
  "scope": {"catalog": "main", "schema": "bronze", "table": "events_raw"},
  "rule": {"type": "retention", "days": 90, "action": "delete"},
  "evidence": ["retention_run_log", "catalog_certification"],
  "review_by": "2027-01-15"
}
```

## What is the difference between governance and security?

| Concern | Security | Governance |
|---|---|---|
| Primary question | who can access it? | who owns it, what does it mean, how does it change? |
| Controls | IAM, encryption, network | catalog, glossary, contracts, retention |
| Failure mode | breach | wrong numbers, untraceable data, compliance gaps |
| Owner | security / CISO | data owner plus platform team |
| Evidence | access logs, pen tests | lineage, contracts, access reviews |

They meet at **classification**: governance decides a column is `pii`, who owns it, and how long it is kept; security turns that into a grant, mask, or row filter. Without classification security has nothing to enforce; without enforcement classification is only a label. Stewards should propose tags (they know what fields mean); security owns the enforcement mechanism.

## What are common roles in data governance (owner/steward/custodian)?

| Role | Accountable for | Typical task |
|---|---|---|
| Data owner | the dataset and its business meaning | approves a breaking change |
| Data steward | quality, definitions, classification | fixes a mislabelled column |
| Data custodian | storage, pipelines, backups, access plumbing | applies the grant |
| Data consumer | following the contract | reports a definition dispute |
| Privacy officer | PII/PHI compliance, erasure requests | signs off retention |

The most common failure is an ownership gap at a team boundary, for example a table produced by a pipeline team but consumed as a finance metric. Name the accountable owner in the catalog, and make sure that owner has authority to approve or block producer changes; ownership without that power is just a name in a table.

## What is a data contract?

A data contract is an agreement between producer and consumers about schema, semantics, quality, SLAs, ownership, and the change process. It is only a contract when a check can fail: the producer versions the contract beside the table, CI compares the produced schema and data against it and blocks the deploy on mismatch, and the catalog publishes it so consumers see what they depend on. A useful contract covers schema, semantics (units, enums, timezone), freshness/SLA, volume bounds, ownership, and change policy.

```yaml
# contracts/orders_v1.yaml
dataset: gold.fact_orders
owner: orders-platform@example.com
sla: {freshness_minutes: 30, availability: "99.9%"}
volume: {expected_rows_per_day: {min: 800000, max: 1400000}}
schema:
  - {name: order_id, type: string, required: true, unique: true}
  - {name: amount, type: "decimal(12,2)", required: true, constraints: ["amount >= 0"]}
  - {name: order_status, type: string, required: true, enum: [created, paid, cancelled, refunded]}
change_policy:
  breaking: requires_new_version_and_consumer_signoff
  additive: allowed_with_notice
consumers:
  - {name: finance_marts, contact: finance-data@example.com}
```

**Change types.** Adding a nullable column is usually backward compatible. Renaming a column, narrowing a type, or making a nullable column required breaks readers. If a producer renames `amount`, the check fails with a diff and the producer either keeps `amount` for a deprecation window or ships v2 and coordinates with the consumers lineage finds. Keeping the contract in a wiki and the schema in the pipeline guarantees drift. Start with one high-value producer-consumer pair, then expand.

This page owns contracts. The runtime checks that enforce them are in [Data quality](data-quality.ipynb), and schema-registry compatibility for streaming producers is in [Data system design](system-design.ipynb).

The cell below runs contract checks against a DuckDB table, the same queries a CI job would run against the warehouse.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
create table fact_orders (order_id varchar, amount decimal(12, 2), order_status varchar, region varchar);
insert into fact_orders values
  ('O1', 120.00, 'paid', 'EU'), ('O2', 75.50, 'created', 'US'),
  ('O3', -5.00, 'paid', 'APAC'), ('O4', 40.00, 'shipped', 'EU'), (null, 10.00, 'paid', 'US');
""")

CONTRACT_CHECKS = {
    "order_id_not_null": "select count(*) from fact_orders where order_id is null",
    "order_id_unique": "select count(*) - count(distinct order_id) from fact_orders where order_id is not null",
    "amount_non_negative": "select count(*) from fact_orders where amount < 0",
    "status_in_enum": """select count(*) from fact_orders
                         where order_status not in ('created', 'paid', 'cancelled', 'refunded')""",
}
violations = {name: con.execute(sql).fetchone()[0] for name, sql in CONTRACT_CHECKS.items()}
print(violations)

failed = sorted(name for name, count in violations.items() if count)
print("deploy blocked by:", failed)
assert failed == ["amount_non_negative", "order_id_not_null", "status_in_enum"]

## How do lineage and cataloging help governance?

The **catalog** says what exists, who owns it, what fields mean, and how to request access. **Lineage** says where data came from (upstream: the incident question) and what depends on it (downstream: the schema-change question). Together they drive impact analysis, audits, PII propagation, cost attribution, and deprecation.

+ **Table-level lineage**: fast blast-radius analysis for schema changes and outages.
+ **Column-level lineage**: needed for PII propagation ("which derived columns contain `customer_email`?").

Metadata arrives two ways: pull-based crawlers read information schemas on a schedule (simple, always lagging), and push-based integrations emit events when jobs write (OpenLineage style, near real time). State lineage freshness when presenting an impact analysis: a table created this morning has no edges in a nightly crawl.

```sql
-- Databricks: downstream columns that read a source column (before dropping it)
select distinct target_table_full_name, target_column_name
from system.access.column_lineage
where source_table_full_name = 'main.silver.customers' and source_column_name = 'email';
```

## What is a data catalog and what should it contain?

Minimum useful set: asset name, location, owner, steward, description, schema, classification tags, freshness SLA, retention, certification status. Operational metadata (last refresh, query counts, downstream usage) is what makes search rank by relevance and separates a live catalog from a stale wiki.

```yaml
asset: main.gold.fact_orders
description: "One row per completed order, conformed for finance reporting."
owner: orders-platform@example.com
steward: finance-data@example.com
classification: [confidential]
tags: [tier-1, finance-certified]
freshness_sla_minutes: 30
retention_days: 2555
schema:
  - {name: order_id, type: string, pii: false}
  - {name: customer_token, type: string, pii: true, policy: tokenized}
```

**Certification** turns a catalog into a trust signal: mark tier-1 datasets certified only after contract tests, ownership, and freshness SLAs exist. Measure coverage as a percentage of tier-1 assets and discover new tables automatically rather than cataloguing only what someone asked about.

## What is business glossary vs technical metadata?

A **glossary** defines business terms ("net revenue", "active user") with an approved definition and an owner. **Technical metadata** describes physical objects (schemas, types, partitions, lineage). Link them: the term points at the metric definition and the columns that implement it, so the definition can be tested against real data. Glossary without technical metadata drifts; technical metadata without a glossary is a schema nobody agrees on. One term, one owner; two glossaries turn the catalog into an argument.

```yaml
term: net_revenue
owner: finance-analytics@example.com
definition: gross_revenue - refunds - chargebacks, excluding tax and internal test orders
mapped_assets: [main.gold.fact_orders.amount, main.gold.fact_refunds.amount]
synonyms: [net sales, net bookings]
```

## RBAC vs ABAC: what is the difference?

**RBAC** grants privileges to roles and assigns users to roles. **ABAC** evaluates policies on attributes of the caller (department, region, purpose) and of the resource (tags such as `classification`, `region`). Purpose-based access is ABAC with the declared purpose as one more attribute.

| Dimension | RBAC | ABAC |
|---|---|---|
| Granularity | table / schema | row, column, attribute |
| Scale | role explosion | policy count stays small |
| Audit | simple to explain | needs policy evaluation logs |
| Best for | coarse access, small orgs | multi-tenant, regional, regulated data |

Modern lakehouses combine them: RBAC grants the base privilege; ABAC (row filters, column masks, tag policies) refines what comes back. A role grant alone is not enough: `select` on a table with no attached policy exposes every row and column.

## What is row-level and column-level security?

**Row-level security** restricts which rows a caller sees (only their region or tenant); the engine injects the filter as an extra predicate. **Column-level security** restricts or masks columns (masked email, hidden salary); a mask function returns the value the caller may see. Together they give cell-level control. Both should run in the query engine, as the definer, so BI, notebooks, and ad-hoc SQL inherit the same policy; masking only in a BI semantic layer leaves a hole for direct SQL.

Where it leaks: joins to an unfiltered copy, clones or extracts created before the policy was attached, and views built with stronger definer rights that re-expose the masked column. Attach policies at schema level and scan for copies.

The cell below simulates engine-enforced policies in DuckDB: callers are rows in a session table, and a secure view applies a region row filter plus an email mask. The test runs as each restricted identity and asserts what it sees, which is how you should test these controls in CI.

In [ ]:
con.execute("""
create table customers (customer_id varchar, email varchar, region varchar);
insert into customers values
  ('C1', 'ana@example.com', 'EU'), ('C2', 'bo@example.com', 'US'), ('C3', 'chen@example.com', 'APAC');
create table principals (principal varchar, region varchar, groups varchar[]);
insert into principals values
  ('analyst_eu', 'EU', ['analysts']), ('privacy_officer', '*', ['analysts', 'pii_readers']);
create table session (principal varchar);
create view customers_secure as
select c.customer_id,
       case when list_contains(p.groups, 'pii_readers') then c.email
            else '***@' || split_part(c.email, '@', 2) end as email,      -- column mask
       c.region
from customers c, session s join principals p using (principal)
where p.region = '*' or c.region = p.region;                              -- row filter
""")


def query_as(principal):
    con.execute("delete from session")
    con.execute("insert into session values (?)", [principal])
    return con.execute("select * from customers_secure order by customer_id").fetchall()


eu_rows, officer_rows = query_as("analyst_eu"), query_as("privacy_officer")
print("analyst_eu      :", eu_rows)
print("privacy_officer :", officer_rows)

assert {row[2] for row in eu_rows} == {"EU"}                  # no other region leaks
assert all(row[1].startswith("***@") for row in eu_rows)      # email masked
assert len(officer_rows) == 3 and officer_rows[0][1] == "ana@example.com"

## How do you implement row-level and column-level security in practice?

Unity Catalog (Databricks): filter and mask functions attached to tables.

```sql
create or replace function main.security.region_filter(region string)
returns boolean
return is_account_group_member('global_readers') or region = current_user_region();
alter table main.gold.fact_orders set row filter main.security.region_filter on (region);

create or replace function main.security.mask_email(email string)
returns string
return case when is_account_group_member('pii_readers') then email
            else concat('***', right(email, 4)) end;
alter table main.silver.customers alter column email set mask main.security.mask_email;
```

Snowflake: masking policies and row access policies.

```sql
create or replace masking policy mask_email as (val string) returns string ->
  case when current_role() in ('PII_READERS') then val
       else regexp_replace(val, '(^[^@]+)@', '****@') end;
alter table silver.customers modify column email set masking policy mask_email;

create or replace row access policy rap_region as (region string) returns boolean ->
  exists (select 1 from security.role_region m
          where m.role_name = current_role() and m.region = region);
alter table gold.fact_orders add row access policy rap_region on (region);
```

BigQuery: column-level security through **policy tags** from a Data Catalog taxonomy attached to columns, and row-level security through `CREATE ROW ACCESS POLICY ... GRANT TO (...) FILTER USING (region = 'EU')`.

Test the control, not the policy text: a CI job assumes each restricted role and fails if an APAC analyst sees a non-APAC row or a masked value appears in clear text. Audit who altered a policy and when.

## How do you handle PII/PHI data in analytics platforms?

A layered approach:

```text
1. Discover and classify: scan for emails, national IDs, card numbers, health codes.
2. Minimize: don't land raw PII in the lake; tokenize at the application boundary.
3. Isolate: a short-retention vault holds raw PII, exempt from broad grants.
4. Protect: encryption at rest/in transit, column masks, row filters, tag-based policies.
5. Govern: retention, erasure, access reviews, audit logs on all of the above.
6. Propagate: column-level lineage flags derived columns that still carry PII.
```

**PHI** is personal data in a health context, so a column can be PII in one table and PHI in another. HIPAA adds minimum-necessary access, business associate agreements, breach notification, and stricter audit retention; PHI usually gets its own classification and an approval workflow rather than self-service access.

**Discovery mechanics.** A scanner samples rows, applies recognizers (regexes, checksums like Luhn for card numbers, dictionaries, ML models), and emits a candidate tag with confidence and the evidence (sample size, match count). A human or second signal confirms before the tag becomes governed: a false positive that masks a needed field breaks analytics, and a column of order numbers can look like a national ID. Scan semi-structured data too; PII hides in free-text tickets, JSON payloads, and logs.

In [ ]:
import re

EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
CARD = re.compile(r"\b(?:\d[ -]?){13,16}\b")


def luhn_valid(candidate: str) -> bool:
    digits = [int(d) for d in re.sub(r"\D", "", candidate)]
    checksum = 0
    for index, digit in enumerate(reversed(digits)):
        if index % 2 == 1:
            digit = digit * 2 - 9 if digit > 4 else digit * 2
        checksum += digit
    return checksum % 10 == 0


def classify(samples, min_rate=0.2):
    """Propose a tag for a column from sampled values, with the evidence a reviewer needs."""
    email_hits = sum(bool(EMAIL.search(value)) for value in samples)
    card_hits = sum(any(luhn_valid(m.group()) for m in CARD.finditer(value)) for value in samples)
    for tag, hits in (("pii_email", email_hits), ("pci_card", card_hits)):
        if hits / len(samples) >= min_rate:
            return {"tag": tag, "hits": hits, "sampled": len(samples)}
    return None


columns = {
    "contact": ["ana@example.com", "n/a", "bo@example.org", "call me"],
    "note": ["paid with 4111 1111 1111 1111", "card 4111 1111 1111 1112", "ok", "refund"],
    "order_ref": ["1234567890123", "9876543210987", "5555555555554", "1111222233334"],
}
proposals = {name: classify(values) for name, values in columns.items()}
for name, proposal in proposals.items():
    print(f"{name:<10} -> {proposal}")

assert proposals["contact"]["tag"] == "pii_email"
assert proposals["note"] == {"tag": "pci_card", "hits": 1, "sampled": 4}   # Luhn rejects the typo
# false positive: order numbers that happen to pass Luhn (about 1 in 10 do), so a human confirms
assert proposals["order_ref"]["tag"] == "pci_card"

## What is data masking and tokenization?

| Technique | Reversible | Stable across rows | Use case |
|---|---|---|---|
| Static masking | no | n/a | non-production copies |
| Dynamic masking | no (raw stored) | n/a | role-based display in production |
| Unsalted hash | no, but brute-forceable | yes | avoid for low-entropy values like emails |
| Keyed hash (HMAC) | no | yes, per key | pseudonymous join key for analytics |
| Tokenization (vault) | yes, via vault | yes | stable key plus controlled re-identification |
| Format-preserving encryption | yes, with key | yes | payment data where format must stay valid |

Dynamic masking is evaluated by the engine on every access, so every reader inherits it. Tokenization replaces the value with a stable surrogate (usually an HMAC with a secret key in a secret manager); joins keep working on the token while the raw value never reaches the analytics layer, and the detokenization mapping lives in a tightly held vault.

Interview cautions: unsalted `sha256(email)` is reversible by hashing a list of known emails, so always use a keyed HMAC. Decide up front whether tokens must survive key rotation, since rotating the key breaks historical joins. **Crypto-shredding** uses a per-subject key: destroying the key makes every token for that subject unlinkable, which helps erasure on append-only data.

In [ ]:
import hashlib
import hmac

TOKEN_KEY = b"demo-key-load-from-a-secret-manager"   # never hard-code a real key


def tokenize(value: str, key: bytes = TOKEN_KEY) -> str:
    return hmac.new(key, value.strip().lower().encode(), hashlib.sha256).hexdigest()[:32]


orders = [("O1", "Ana@Example.com"), ("O2", "bo@example.com"), ("O3", "ana@example.com ")]
tokenized = [(order_id, tokenize(email)) for order_id, email in orders]
for row in tokenized:
    print(row)

assert tokenized[0][1] == tokenized[2][1]                         # same customer joins across rows
assert tokenized[0][1] != tokenized[1][1]
assert tokenize("ana@example.com", b"rotated-key") != tokenized[0][1]   # rotation breaks old joins

# unsalted hashes fall to a dictionary attack over likely emails
leaked_hash = hashlib.sha256(b"bo@example.com").hexdigest()
candidates = ["ana@example.com", "bo@example.com", "chen@example.com"]
assert next(c for c in candidates if hashlib.sha256(c.encode()).hexdigest() == leaked_hash) == "bo@example.com"

De-identifying a research extract follows the same idea in SQL: replace direct identifiers with keyed tokens, shift dates, and keep only the fields the purpose needs (minimum necessary). In production the shift is a secret per-patient offset so intervals between a patient's events are preserved.

In [ ]:
con.execute("""
create table encounters (patient_mrn varchar, patient_name varchar, admit_date date, diagnosis_code varchar);
insert into encounters values
  ('MRN-1', 'Ana Silva', '2026-03-14', 'E11.9'), ('MRN-1', 'Ana Silva', '2026-04-02', 'I10'),
  ('MRN-2', 'Bo Chen',   '2026-05-20', 'J45.909');
create table patient_offsets as
select patient_mrn, cast(17 + row_number() over (order by patient_mrn) * 11 as integer) as shift_days
from (select distinct patient_mrn from encounters);
create view encounters_deidentified as
select sha256('demo-salt|' || e.patient_mrn) as patient_token,
       e.admit_date - o.shift_days         as admit_date_shifted,
       e.diagnosis_code
from encounters e join patient_offsets o using (patient_mrn);
""")
deid = con.execute("select * from encounters_deidentified order by patient_token, admit_date_shifted").fetchall()
for row in deid:
    print(row[0][:12], row[1], row[2])

columns = [c[0] for c in con.execute("describe encounters_deidentified").fetchall()]
assert "patient_name" not in columns and "patient_mrn" not in columns
visits = {}
for token, shifted, _ in deid:
    visits.setdefault(token, []).append(shifted)
first_visit, second_visit = next(v for v in visits.values() if len(v) == 2)
assert (second_visit - first_visit).days == 19       # interval between visits preserved

## How do you tag and classify data in a catalog?

Tags bridge governance intent and enforcement. Use a small, controlled taxonomy (`public < internal < confidential < restricted`, with restricted covering PII, PHI, PCI, secrets) defined once at account level, with a fixed set of allowed values and a small group allowed to assign them. Free-form tags drift: two spellings of `pii` mean one is invisible to the policy engine. Tags should propagate: table tags default to columns, and lineage-aware scanners copy tags onto derived columns. Tag at creation time in the pipeline and run a daily job listing unclassified columns.

```sql
-- Unity Catalog
alter table main.silver.customers
alter column email set tags ('pii_type' = 'email', 'classification' = 'restricted');
alter table main.gold.fact_orders set tags ('domain' = 'sales', 'tier' = '1');

select column_name, tag_name, tag_value
from system.information_schema.column_tags
where catalog_name = 'main' and schema_name = 'silver' and table_name = 'customers';
```

## How do you implement tag-based access control (Unity Catalog, Lake Formation)?

Tag-based policies match tags rather than object names, so a new table is covered as soon as it is tagged. That is the point of ABAC, and also its failure mode: an untagged sensitive table matches no policy and the default grant applies.

```sql
-- Unity Catalog ABAC: mask every column tagged pii_type, except for pii_readers
create policy main.security.pii_column_policy
on catalog main
as
  for select to all except (pii_readers)
  match columns with tag pii_type
  using column mask main.security.mask_email;
```

**AWS Lake Formation LF-Tags**: a tag key with allowed values (`classification = public | internal | confidential | restricted`) is attached to databases, tables, or columns, and principals are granted by tag expression (`analyst` may `SELECT` where `classification = public`). Tables tagged later are covered immediately. Row and cell-level security is a separate feature (**data filters**) attached to a grant; most restricted tables need both. Inspect effective permissions with `aws lakeformation list-permissions` and monitor access in CloudTrail.

```hcl
resource "aws_lakeformation_lf_tag" "classification" {
  key    = "classification"
  values = ["public", "internal", "confidential", "restricted"]
}
```

## What is policy as code for data governance?

Governance rules live in git, are reviewed like software, and are applied by automation. A pull request runs unit tests asserting allow/deny decisions, a plan step shows the diff, and an apply step uses a service identity that can only write policy objects. A scheduled drift job compares live policy with committed policy, because console edits otherwise make the repository a stale snapshot. Temporary exceptions are commits with an expiry date that the drift job enforces. The result is reviewable diffs, consistent enforcement across engines, and an audit trail (author, reason, timestamp) that is exactly what auditors ask for.

```yaml
# policies/pii_access.yaml
kind: DataPolicy
metadata: {name: pii-read-access}
spec:
  match: {tags: {pii_type: "*"}}
  rules:
    - {effect: deny, principals: {not_in_group: pii_readers}, actions: [select, export]}
    - effect: allow
      principals: {in_group: pii_readers}
      actions: [select]
      conditions: [{mfa_required: true}, {purpose: [fraud_investigation, compliance]}]
  exceptions: {process: access-request-ticket, max_duration_days: 30}
```

In [ ]:
from datetime import date


def evaluate(subject, action, resource, exceptions=(), today=date(2026, 11, 6)):
    """Deny by default; deny-rules win over allow-rules; exceptions expire."""
    active = any(
        e["principal"] == subject["name"] and e["resource"] == resource["name"] and e["expires"] >= today
        for e in exceptions
    )
    if "pii_type" in resource["tags"] and "pii_readers" not in subject["groups"] and not active:
        return "deny"
    if action in {"export", "download"} and resource["tags"].get("classification") == "restricted":
        return "deny"
    if subject["team"] in resource["allowed_teams"] or active:
        return "allow"
    return "deny"


customers = {"name": "silver.customers", "tags": {"pii_type": "email", "classification": "restricted"},
             "allowed_teams": {"growth", "privacy"}}
analyst = {"name": "alice", "groups": [], "team": "growth"}
officer = {"name": "omar", "groups": ["pii_readers"], "team": "privacy"}
migration = [{"principal": "alice", "resource": "silver.customers", "expires": date(2026, 10, 31)}]

assert evaluate(analyst, "select", customers) == "deny"
assert evaluate(officer, "select", customers) == "allow"
assert evaluate(officer, "export", customers) == "deny"
assert evaluate(analyst, "select", customers, exceptions=migration) == "deny"     # expired exception
assert evaluate(analyst, "select", customers, exceptions=migration, today=date(2026, 10, 15)) == "allow"
print("policy decision tests passed")

## How do you implement auditability for data access?

Log access events (principal, action, object, columns, timestamp, source address, often query text), export and download events, permission and policy changes, retention and erasure actions, and admin actions. Copy them to immutable storage with a retention lock that outlives the review cycle and that the audited administrators cannot delete. Alert on rare events: new grants, exports to new locations, access from unfamiliar networks.

```sql
-- Unity Catalog audit events
select event_time, user_identity.email as actor, action_name,
       request_params.full_name_arg as table_name, source_ip_address
from system.access.audit
where action_name in ('getTable', 'commandSubmit', 'updatePermissions')
  and event_time > current_timestamp() - interval 7 days;

-- Snowflake access history for a sensitive table
select h.query_start_time, h.user_name, o.value:objectName::string as object_name
from snowflake.account_usage.access_history h,
     lateral flatten(input => h.direct_objects_accessed) o
where o.value:objectName::string = 'SILVER.CUSTOMERS';
```

## How do you run access reviews and certifications?

On a cadence, list every principal, their entitlements, and when each was last used; the owner certifies keep or revoke; the system records decision, reviewer, and date and removes revoked access automatically. That record is the audit evidence. Usage is the key input: an entitlement unused for 180 days, especially a direct grant rather than a role, is the cheapest risk to remove. Bulk one-click approvals produce evidence that nobody looked; sample them and report the sample size.

```json
{"campaign": "2026-Q4-sensitive-access", "reviewer": "data-council",
 "decisions": [
   {"principal": "analyst_apac", "entitlement": "gold.fact_orders", "decision": "keep", "reason": "regional reporting"},
   {"principal": "contractor_17", "entitlement": "gold.fact_orders", "decision": "revoke", "reason": "no access in 180 days"}]}
```

## How do you handle GDPR right to be forgotten in a lakehouse or warehouse?

```text
Erasure runbook
1. Resolve the subject to every identifier (customer_id, email token, device id).
2. Use lineage to find every table carrying those keys: raw, derived marts, features, caches.
3. Vault: delete the raw PII mapping (or destroy the per-subject key: crypto-shredding).
4. Bronze/silver/gold: delete keyed rows; rebuild aggregates that included them.
5. Downstream: purge search indexes, caches, exports, ML feature stores.
6. Physically remove: expire table snapshots / vacuum old files so time travel cannot resurrect the row.
7. Evidence: immutable erasure record (request id, scope, operator, timestamp, per-table receipt).
8. Backups: rely on documented backup expiry; don't promise instant backup erasure.
```

```sql
-- Iceberg: targeted deletes, then expire snapshots that still reference the rows
delete from main.bronze.events_raw where customer_token = :token;
delete from main.silver.customers_current where customer_token = :token;
call main.system.expire_snapshots(table => 'bronze.events_raw',
                                  older_than => timestamp '2026-10-04 00:00:00', retain_last => 1);
-- Delta: DELETE ... then VACUUM table RETAIN 168 HOURS
```

A delete on a table with time travel or an immutable raw log is not erasure until old snapshots and files are expired. State the guarantee precisely: "new queries cannot see it immediately; physical removal completes at the next expiry, within N days." Deleting only from the serving layer leaves the raw copy, backups, and snapshots holding the value. Storing PII in a separate keyed vault (or tokenizing at ingestion) turns erasure into a keyed delete instead of a rewrite of immutable files.

## What are data retention policies and how do you enforce them?

A retention policy names the data class, the period, the end-of-life action (delete, archive, anonymize), and the legal basis. Enforcement is a scheduled job that drops or deletes partitions and expires snapshots, plus object-storage lifecycle rules for files. Each run records counts, and an alert fires when any partition is older than its policy allows; a policy with no run record is just a note. A **legal hold** overrides deletion and must be recorded.

```yaml
retention:
  - {dataset: main.bronze.events_raw, days: 90,   action: delete,  basis: storage_limitation}
  - {dataset: main.silver.orders,     days: 2555, action: archive, basis: tax_law}
  - {dataset: audit_logs,             days: 3650, action: archive_immutable}
  - {dataset: backups,                days: 35,   action: expire}
```

Retention and erasure interact: shorter raw retention shrinks the erasure surface but shortens replay ability. State the trade-off and who owns it (privacy officer plus platform owner).

In [ ]:
from datetime import timedelta

con.execute("""
create table events_raw (event_date date, customer_token varchar, payload varchar);
insert into events_raw
select date '2026-11-06' - cast(i as integer), 'tok' || (i % 3), 'event'
from range(0, 120, 10) t(i);
create table retention_policy (dataset varchar, days integer, legal_hold boolean);
insert into retention_policy values ('events_raw', 90, false);
create table retention_runs (run_date date, dataset varchar, rows_deleted bigint);
""")


def enforce_retention(run_date):
    days, hold = con.execute(
        "select days, legal_hold from retention_policy where dataset = 'events_raw'"
    ).fetchone()
    if hold:
        return 0
    cutoff = run_date - timedelta(days=days)
    deleted = con.execute("select count(*) from events_raw where event_date < ?", [cutoff]).fetchone()[0]
    con.execute("delete from events_raw where event_date < ?", [cutoff])
    con.execute("insert into retention_runs values (?, 'events_raw', ?)", [run_date, deleted])
    return deleted


deleted = enforce_retention(date(2026, 11, 6))
oldest = con.execute("select min(event_date) from events_raw").fetchone()[0]
print("deleted:", deleted, "| oldest remaining:", oldest, "| run log:", con.execute("from retention_runs").fetchall())
assert deleted == 2 and (date(2026, 11, 6) - oldest).days <= 90

# erasure for one subject: keyed delete plus a receipt
before = con.execute("select count(*) from events_raw where customer_token = 'tok1'").fetchone()[0]
con.execute("delete from events_raw where customer_token = 'tok1'")
assert before > 0 and con.execute("select count(*) from events_raw where customer_token = 'tok1'").fetchone()[0] == 0

## How do you measure data governance maturity?

Governance needs outcome metrics, or it becomes an unenforceable document. Report coverage and enforcement, not activity (policies written says nothing about exposure), and track the same figures over time.

+ **Coverage**: percent of tier-1 datasets with owner, definition, freshness SLA.
+ **Classification**: percent of columns classified; median time to classify a new column.
+ **Enforcement**: percent of sensitive columns with a mask or policy.
+ **Access hygiene**: orphan or unused grants, over-privileged service accounts, reviews done on time.
+ **Contracts**: CI checks passing; incidents from unannounced schema changes.
+ **Lifecycle**: datasets past retention without exemption; erasure requests closed within SLA.

| Level | Signal | Example metric |
|---|---|---|
| Ad hoc / reactive | tickets drive fixes, definitions in chat | time to classify a new column |
| Documented / managed | catalog, owners, SLAs exist | percent of tier-1 datasets with an owner |
| Enforced | tags drive policies, contracts in CI, retention automated | percent of sensitive columns masked |
| Optimized | policy as code, measured reviews, exceptions expire | policy drift detected within a day |

## How do you operationalize governance at scale?

At small scale governance is a person and a spreadsheet; at scale it is a set of automated loops, each with an owner, a cadence, and a measurable output, all sharing data:

+ **Policy as code**: define in git, test decisions in CI, plan, apply with a scoped identity, detect drift.
+ **Classification**: sample new or changed tables, run recognizers, queue proposals with evidence, confirm into governed tags, measure the false-positive rate.
+ **Lineage-driven change management**: a proposed change finds downstream assets within N hops, notifies owners ranked by tier and usage (a raw count of 300 downstream tables gets ignored), and blocks until acknowledged or made compatible.
+ **Access reviews**: entitlements plus last use, certified by owners, revocations applied automatically.

Classification feeds the tags policies match; lineage feeds change review and erasure; audit events feed access reviews. When one loop is missing, the others quietly degrade.

## Real world use cases for data governance

| Scenario | Design | Why it holds |
|---|---|---|
| E-commerce, 30M customers, ~500 erasure requests/month | raw PII in a short-retention vault; lakehouse stores `customer_token`; `pii_type` tags drive a mask policy; erasure orchestrator deletes tokens and writes an immutable receipt | erasure is a keyed delete, not a rewrite; new columns inherit protection without tickets |
| Fintech, EU/US/APAC tenants, PCI data | every table carries `region` and `tenant_id`; composed row filters; tokenized card columns; tag-driven policies | enforced in the engine for every client; tenant isolation holds inside a shared region |
| SaaS, 2,000 tenant schemas, past rename outage | contracts in git validated in CI and published to the catalog; schema registry enforces `BACKWARD` on topics | "please tell us before renaming" becomes a build failure |
| Hospital network, HIPAA, 7-year audit | PHI in a restricted schema with its own approval flow; researchers use a de-identified view with tokens and date shifting; immutable access logs; automated retention | researchers never touch PHI; evidence comes from logs, not policy PDFs |
| Acquisition, 12,000 tables to merge, 300 to deprecate | automated profiling and classification; column-level lineage from dashboards to sources; owner, deprecation date, and replacement per legacy table; weekly remaining-consumer report | deprecation driven by evidence and deadlines instead of accidental outages |

## Data governance exercises

Small standard-library calculations behind common governance decisions. Each states the task and the expected result, then asserts it.

### Try it 1. Classification coverage

**Task.** Compute coverage weighted over all columns and the unweighted mean of per-table coverage; find the worst table. **Expected:** weighted ~38.7%, unweighted ~47.4%, worst `events`. Report the weighted figure: the unweighted mean lets small tables hide large gaps.

In [ ]:
tables = [
    {"name": "customers", "columns": 120, "classified": 96},
    {"name": "orders", "columns": 85, "classified": 40},
    {"name": "events", "columns": 240, "classified": 36},
]

weighted = sum(t["classified"] for t in tables) / sum(t["columns"] for t in tables)
unweighted = sum(t["classified"] / t["columns"] for t in tables) / len(tables)
worst = min(tables, key=lambda t: t["classified"] / t["columns"])

print(f"weighted = {weighted:.1%}, unweighted = {unweighted:.1%}, worst = {worst['name']}")
assert round(weighted, 3) == 0.387
assert abs(unweighted - 0.4735) < 1e-3
assert worst["name"] == "events"

### Try it 2. Sample size for PII discovery

**Task.** Rows to sample so a value present in a fraction `p` of rows is seen at least once with confidence `c`: solve `(1 - p) ** n = 1 - c`. **Expected:** 299 rows at 1% prevalence, 2,995 at 0.1% (95% confidence).

In [ ]:
import math


def sample_size(prevalence, confidence):
    return math.ceil(math.log(1 - confidence) / math.log(1 - prevalence))


for rate in (0.01, 0.001):
    print(f"prevalence {rate:.1%} -> {sample_size(rate, 0.95)} rows")
assert sample_size(0.01, 0.95) == 299
assert sample_size(0.001, 0.95) == 2995
assert sample_size(0.01, 0.99) > sample_size(0.01, 0.95)

### Try it 3. Cost of erasure without an index on subject keys

**Task.** 40M rows/day, 90-day retention, 500 erasure requests/day, 12 rows per subject, full scans at 50M rows/s, deletes at 5,000 rows/s. **Expected:** 72 s scan per request, 10 hours of scanning per day, versus 1.2 s of actual deletes, so index or cluster by subject key.

In [ ]:
daily_rows, retention_days, requests_per_day, rows_per_subject = 40_000_000, 90, 500, 12
scan_rows_per_second, erase_rows_per_second = 50_000_000, 5_000

scan_seconds_per_request = daily_rows * retention_days / scan_rows_per_second
scan_hours_per_day = requests_per_day * scan_seconds_per_request / 3600
erase_seconds_per_day = requests_per_day * rows_per_subject / erase_rows_per_second

print(f"{scan_seconds_per_request:.0f}s per request, {scan_hours_per_day:.1f}h/day scanning, "
      f"{erase_seconds_per_day:.1f}s/day deleting")
assert scan_seconds_per_request == 72.0
assert round(scan_hours_per_day, 1) == 10.0
assert erase_seconds_per_day == 1.2

### Try it 4. Contract violation from a schema diff

**Task.** Report added, removed, and type-changed columns; removals and type changes are breaking. **Expected:** `region` added, `amount` type changed, breaking.

In [ ]:
expected = {"order_id": "string", "amount": "decimal(18,2)", "status": "string"}
actual = {"order_id": "string", "amount": "double", "status": "string", "region": "string"}


def schema_diff(expected, actual):
    return {
        "added": sorted(set(actual) - set(expected)),
        "removed": sorted(set(expected) - set(actual)),
        "type_changed": sorted(n for n in set(expected) & set(actual) if expected[n] != actual[n]),
    }


result = schema_diff(expected, actual)
breaking = bool(result["removed"] or result["type_changed"])
print(result, "breaking:", breaking)
assert result == {"added": ["region"], "removed": [], "type_changed": ["amount"]}
assert breaking

### Try it 5. Lineage blast radius

**Task.** Breadth-first search from `raw.orders` over "is read by" edges, recording the depth at which each asset is first reached. **Expected:** 6 downstream assets, `dash.revenue` at depth 3.

In [ ]:
from collections import deque

edges = [
    ("raw.orders", "stg.orders"), ("stg.orders", "fct.orders"),
    ("fct.orders", "dash.revenue"), ("fct.orders", "ml.features"),
    ("raw.orders", "stg.customers"), ("stg.customers", "dim.customers"),
    ("dim.customers", "dash.revenue"),
]


def blast_radius(edges, source):
    adjacency = {}
    for src, dst in edges:
        adjacency.setdefault(src, []).append(dst)
    depth = {source: 0}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for nxt in adjacency.get(node, []):
            if nxt not in depth:
                depth[nxt] = depth[node] + 1
                queue.append(nxt)
    depth.pop(source)
    return depth


radius = blast_radius(edges, "raw.orders")
print(sorted(radius.items(), key=lambda item: (item[1], item[0])))
assert len(radius) == 6 and radius["dash.revenue"] == 3 and max(radius.values()) == 3

### Try it 6. Score governance maturity

**Task.** Weighted score over ownership, classification, enforcement, and audit; map to optimized (>= 0.80), enforced (>= 0.60), managed (>= 0.40), else reactive. **Expected:** 0.7675, `enforced`, next fix `enforcement`.

In [ ]:
dimensions = {
    "ownership": {"weight": 0.30, "score": 0.90},
    "classification": {"weight": 0.25, "score": 0.75},
    "enforcement": {"weight": 0.25, "score": 0.60},
    "audit": {"weight": 0.20, "score": 0.80},
}
LEVELS = [(0.80, "optimized"), (0.60, "enforced"), (0.40, "managed"), (0.0, "reactive")]

assert round(sum(d["weight"] for d in dimensions.values()), 2) == 1.0
score = sum(d["weight"] * d["score"] for d in dimensions.values())
level = next(name for threshold, name in LEVELS if score >= threshold)
next_fix = min(dimensions, key=lambda name: dimensions[name]["score"])

print(f"score = {score:.4f}, level = {level}, next fix = {next_fix}")
assert round(score, 4) == 0.7675 and level == "enforced" and next_fix == "enforcement"